In [ ]:
"""eod_sale_service — Gold build. Cast the conformed line to the final fact schema
(the 55 business columns; add_audit_columns appends created_at/updated_at, matching the
legacy ClickHouse hq_report.eod_sale_service) and write idempotently with replaceWhere.
"""
from pyspark.sql import functions as F

from ssv_data.io.writers import write_delta
from ssv_data.schema.cast import fill_missing_columns, cast_by_schema, select_schema
from ssv_data.transforms.common import add_audit_columns

# Target fact schema (type tokens). Matches hq_report.eod_sale_service column-for-column;
# report_date is the VN date-part of transaction_time (source time is already local).
FACT_SERVICE_SCHEMA = {
    "report_date": "date", "transaction_id": "str", "transaction_time": "date", "posting_time": "date",
    "store_id": "str", "store_name": "str", "movement_type": "int", "has_invoice": "bool",
    "customer_age_range": "str", "customer_nationality": "str", "customer_gender": "str",
    "edc_type": "int", "cash": "int", "store_type": "str", "service_name": "str",
    "amount_before_vat": "int", "amount_vat": "int", "total_amount_without_promotion": "int",
    "staff_id": "str", "staff_name": "str", "pos_id": "str", "payment_method": "int",
    "payment_method_value": "int", "transaction_record_start_time": "date",
    "transaction_record_end_time": "date", "promotion_total_amount": "int",
    "total_product_promotion_amount_without_tax": "int",
    "total_voucher_promotion_amount_without_tax": "int",
    "total_bill_promotion_amount_without_tax": "int", "service_fee": "int",
    "customer_service_fee": "int", "skip_validate_over_50_percent": "bool",
    "skip_validate_over_50_percent_by_int": "int", "receivable_amount": "int", "pre_paid": "bool",
    "synced_sap_status": "str", "synced_sap_time": "date", "order_no": "str", "service_id": "str",
    "provider_id": "str", "customer_id": "str", "order_type": "str", "product_code": "str",
    "product_name": "str", "product_uom_id": "str", "supplier_code": "str", "supplier_name": "str",
    "quantity": "int", "purchase_price_with_tax": "int", "purchase_price_without_tax": "int",
    "total_amount": "int", "retail_business_type": "str", "commission_on_vnd": "int",
    "total_commission": "int",
}


In [ ]:
def service_gold_build(ctx) -> None:
    df = ctx.spark.read.table(ctx.silver("sale_service_line"))
    df = fill_missing_columns(df, FACT_SERVICE_SCHEMA)
    df = cast_by_schema(df, FACT_SERVICE_SCHEMA)
    df = df.withColumn("report_date", F.to_date("report_date"))   # DateType partition values
    fact = add_audit_columns(select_schema(df, FACT_SERVICE_SCHEMA))
    fact = fact.where(F.col("report_date") == F.to_date(F.lit(ctx.running_date)))
    write_delta(ctx, fact, ctx.gold("fact_eod_sale_service"),
                partition_col="report_date", replace_where=ctx.replace_where)
    ctx.logger.info("gold(service) build complete")
